# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/faisaljaam002-png/flyrank-assignment1/blob/main/work/notebooks/w07_action_playbook.ipynb)

**Lane 2 — Refresh / Content Opportunity Scoring.** This notebook turns the validated output of
w01–w06 into a **content action playbook**: a ranked queue with reason codes, an
archetype→action map, the decay/refresh insight, intended use and limits, a human-review +
no-go list, monitoring/retrain triggers, and cost/value thinking. The playbook's queue, metrics
and figures are exported to `work/outputs/` and `work/figures/` — the exact artifacts the paper's
recommendations section builds on next week.

Skill: `writing-honest-claims` + `flyrank/flyrank-data` (loaded from `skills/README.md`).

> Claim language throughout: **observed / measured / directional / decision-support**. Nothing
> here is causal — no refresh was run, so no sentence says "refreshing will recover traffic."

## 0. Setup (Colab or local)

On Colab this clones the repo and installs requirements. Locally it just moves to the repo root
and loads the starter slice.

In [1]:
import os, sys, subprocess, json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/faisaljaam002-png/flyrank-assignment1"

if IN_COLAB:
    if not os.path.isdir("flyrank-assignment1"):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "flyrank-assignment1"], check=True)
    os.chdir("flyrank-assignment1")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    root = Path(os.getcwd())
    for _ in range(4):
        if (root / "data" / "raw").exists():
            os.chdir(root)
            break
        root = root.parent

print("Working dir:", os.getcwd())
root = Path(os.getcwd())
assert (root / "data/raw/content_refresh_anonymized.csv").exists(), "starter CSV not found"
df = pd.read_csv(root / "data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
df["trend_direction"] = df["trend_direction"].astype(str)
print(f"Loaded {len(df):,} pages, {df['client_id'].nunique()} clients")
print(f"Declining base rate: {df['is_declining_label'].mean():.3f}")

SEED = 2026
rng = np.random.default_rng(SEED)
try:
    plt.style.use("seaborn-v0_8-whitegrid")
except Exception:
    pass
print("Versions: pandas", pd.__version__, "| numpy", np.__version__, "| sklearn", __import__("sklearn").__version__)
print("Random seed fixed:", SEED)

Working dir: E:\FlyRank Ai\flyrank-assignment1


Loaded 30,000 pages, 32 clients
Declining base rate: 0.542
Versions: pandas 3.0.2 | numpy 2.4.4 | sklearn 1.8.0
Random seed fixed: 2026


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

**The decision this playbook supports.** Which content page should an editor review *first*,
given limited review capacity. One row = one page; the output is an ordered queue where every
row carries exactly one score, one reason code, and one action label.

**The validated comparison (recomputed in this run, section 1 code).** On 8 of 32 clients held
out as a group — the honest, grouped-by-client split from w05/w06 — the transparent rule beats
both learned models on the editor's decision metric, Precision@50:

| Method | P@10 | P@50 | test base rate |
|---|---|---|---|
| Transparent rule (w04) | 0.90 | 0.92 | 0.634 |
| Random Forest (w05) | 0.80 | 0.88 | 0.634 |
| Logistic Regression (w05) | 0.80 | 0.86 | 0.634 |

A transparent rule that "cannot lose" at Precision@50 beats models that "learned" — that is the
honest finding (w05/w06), and it decides what the queue is built from: **the rule**, with the
learned ranking kept as a decision-support cross-check, not a replacement.

**The ranked actions, in priority order** (the order an editor works through the queue):

| Priority | Action label | Reason code | Fires when | Why it sits where it sits |
|---|---|---|---|---|
| 1 | `refresh_and_review_ctr` | `ctr_lag_visible` | position 1-20, ≥300 impressions, CTR < 0.5 | The only CONFIRMED signal: declining rate rises monotonically as CTR falls (0.42 → 0.67), and the visible zero-click extreme runs 0.76. Biggest defensible queue. |
| 2 | `refresh` | `stale_visible` | position 1-20, ≥300 impressions, ≥104 days untouched, CTR ≥ 0.5 | Staleness is MIXED: only the far tail (104+ days) is consistently elevated (0.62–0.64 vs ~0.58 base). A gated bonus, never a primary flag. |
| 3 | `watch_and_monitor` | `visible_opportunity` | ≥300 impressions, no urgent flag above | Visible pages still clicking fine — no measured problem; recheck next cycle. |
| 4 | `monitor` | `low_traffic_monitor` | <300 impressions | Below the volume where a decline signal can be measured — the honest answer is "re-measure", not "fix". |

**Archetype → action mapping.** A content team thinks in page types, not formulas. The queue is
the same rule; the archetype column names the page type so the human look is faster. The
archetypes are a **label-free partition** — their declining rates are measured, not defined:

| Archetype | What it looks like | Action |
|---|---|---|
| `zero_click_visible` | visible (≥300 imp), page 1-2, CTR = 0 | `refresh_and_review_ctr` — first look |
| `ctr_lag_visible` | visible, page 1-2, 0 < CTR < 0.5 | `refresh_and_review_ctr` |
| `stale_tail_visible` | visible, page 1-2, ≥104 days, CTR ≥ 0.5 | `refresh` |
| `visible_opportunity` | visible, page 1-2, no urgent flag | `watch_and_monitor` |
| `buried_visible` | visible but past page 2 (or no position) | `watch_and_monitor` |
| `quiet_low_volume` | <300 impressions | `monitor` |

**One advisory flag on top of the map:** within `zero_click_visible`, a subset — the
`zero_ctr_stable_quirk` (page 1-2, CTR = 0, but trend **stable/up**; count printed in section 2)
— keeps its high score but gets **extra skepticism before any edit**. These are the
false-alarm rows the human review step exists for.

**The decay / refresh insight.** The two words of this lane, measured: freshness is **not** a
general risk signal. Pages refreshed 0-30 days ago decline at the same rate as the visible
population (~0.58), and only the 104+ day tail is elevated — so "refresh old pages" as a blanket
rule is not supported by this data. The paper's "freshness multiplier" (3.2× health, 57×
impressions for refreshed 365+ pages) came from a tiny, proxy-heavy bucket (w06 Finding B) and
is not reproduced here. The honest, directional reading: **decay is signaled by the CTR gap at a
good position; refresh is worth prioritizing only at the far staleness tail.**

**Cost / value thinking.** At the held-out P@50 of 0.92 the queue concentrates review effort: 46
of the top-50 are declining, versus ~32 from random order at the 0.634 test base rate. Editor
hours saved were **not** measured (that claim was rewritten in w06). The cost side is the false
alarm: 38% of flagged pages (4,363 of 11,593) are not declining, so a human look is mandatory
and the top-10 review stays manual. The queue sizes *which pages to open*; it does not replace
judgment.

In [2]:
# --- 1a. The validated rule, re-run so the queue is produced in this notebook ---
def pct_rank(col):
    return pd.to_numeric(col, errors="coerce").fillna(0).rank(method="average", pct=True).fillna(0)

def rule_score(sub):
    on12 = (sub["avg_position"] > 0) & (sub["avg_position"] <= 20)
    ctr_gap = on12 * (1 - pct_rank(sub["ctr"]))
    stale = (on12 & (sub["days_since_last_update"] >= 104)).astype(int)
    vol = pct_rank(np.log1p(sub["impressions_90d"]))
    return 0.85 * ctr_gap + 0.10 * stale + 0.15 * vol

on12 = (df["avg_position"] > 0) & (df["avg_position"] <= 20)
visible = df["impressions_90d"] >= 300
low_ctr = on12 & visible & (df["ctr"] < 0.5)
stale_vis = on12 & visible & (df["days_since_last_update"] >= 104)

df["score"] = rule_score(df)
df["reason_code"] = np.select([low_ctr, stale_vis, visible],
                              ["ctr_lag_visible", "stale_visible", "visible_opportunity"],
                              default="low_traffic_monitor")
df["action_label"] = np.select([low_ctr, stale_vis, visible],
                               ["refresh_and_review_ctr", "refresh", "watch_and_monitor"],
                               default="monitor")

# --- 1b. archetype labels (mutually exclusive, label-free partition; quirk is an advisory flag) ---
df["archetype"] = np.select(
    [~visible,
     on12 & visible & (df["ctr"] == 0),
     low_ctr,
     stale_vis,
     on12 & visible,
     visible],
    ["quiet_low_volume", "zero_click_visible", "ctr_lag_visible",
     "stale_tail_visible", "visible_opportunity", "buried_visible"],
    default="other")
# advisory flag: zero-CTR at page 1-2 whose trend is stable/up -> the false-alarm population
df["is_quirk"] = (on12 & visible & (df["ctr"] == 0) & df["trend_direction"].isin(["stable", "up"])).astype(int)

queue = df.sort_values(["score", "impressions_90d"], ascending=[False, False]).reset_index(drop=True)
queue["rank"] = queue.index + 1

# --- 1c. honest held-out comparison (same split as w05/w06, recomputed here) ---
def prec_at_k(ranked_y, k):
    if isinstance(ranked_y, pd.DataFrame):
        ranked_y = ranked_y["y"]
    return ranked_y.head(k).mean()

client_ids = pd.Series(df["client_id"].unique())
holdout = pd.Series(rng.choice(client_ids, size=int(len(client_ids) * 0.25), replace=False))
train_mask = ~df["client_id"].isin(holdout)
test_mask = df["client_id"].isin(holdout)

feat = df.copy()
feat["has_keyword"] = feat["search_volume"].notna().astype(int)
feat["has_word_count"] = feat["word_count"].notna().astype(int)
feat["has_position"] = (feat["avg_position"] > 0).astype(int)
for c in ["impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d"]:
    feat[f"log_{c}"] = np.log1p(feat[c])
NUMERIC_FEATURES = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d",
    "days_with_impressions", "days_with_sessions",
    "content_age_days", "days_since_last_update",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
]
CATEGORICAL_FEATURES = [
    "competition_level", "content_type", "main_intent",
    "age_tier", "freshness_tier", "word_count_tier", "impression_tier", "position_tier",
]
HAS_FLAGS = ["has_keyword", "has_word_count", "has_position"]
for c in NUMERIC_FEATURES:
    feat[c] = pd.to_numeric(feat[c], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)
for c in CATEGORICAL_FEATURES:
    feat[c] = feat[c].fillna("unknown").astype(str).replace({"": "unknown", "nan": "unknown"})
X = feat[NUMERIC_FEATURES + CATEGORICAL_FEATURES + HAS_FLAGS].copy()
y = df["is_declining_label"]

def encode(Xdf):
    return pd.get_dummies(Xdf, columns=CATEGORICAL_FEATURES, drop_first=True)

X_tr_e = encode(X[train_mask])
X_te_e = encode(X[test_mask]).reindex(columns=X_tr_e.columns, fill_value=0)
scaler = StandardScaler().fit(X_tr_e)
X_tr_s = pd.DataFrame(scaler.transform(X_tr_e), columns=X_tr_e.columns, index=X_tr_e.index)
X_te_s = pd.DataFrame(scaler.transform(X_te_e), columns=X_te_e.columns, index=X_te_e.index)

models = {
    "Random Forest": RandomForestClassifier(n_estimators=300, max_depth=6, min_samples_leaf=20, random_state=SEED, n_jobs=-1),
    "Logistic Regression": LogisticRegression(max_iter=2000, random_state=SEED),
}
results = {}
for name, m in models.items():
    fit_in = X_tr_e if name == "Random Forest" else X_tr_s
    val_in = X_te_e if name == "Random Forest" else X_te_s
    m.fit(fit_in, y[train_mask])
    pr = m.predict_proba(val_in)[:, 1]
    rk = pd.DataFrame({"y": y[test_mask].values, "p": pr}).sort_values("p", ascending=False)
    results[name] = {"P@10": prec_at_k(rk, 10), "P@50": prec_at_k(rk, 50), "proba": pr}

test_sub = df[test_mask].reset_index(drop=True).copy()
test_sub["baseline"] = rule_score(test_sub)
base_ranked = test_sub.sort_values(["baseline", "impressions_90d"], ascending=[False, False])
base_y = pd.DataFrame({"y": base_ranked["is_declining_label"].values})
results["Transparent rule (w04)"] = {"P@10": prec_at_k(base_y, 10), "P@50": prec_at_k(base_y, 50), "proba": None}

comparison = pd.DataFrame({"P@10": {k: v["P@10"] for k, v in results.items()},
                           "P@50": {k: v["P@50"] for k, v in results.items()}})
comparison = pd.concat([pd.DataFrame([{"P@10": y[test_mask].mean(), "P@50": y[test_mask].mean()}],
                                     index=["Base rate"]), comparison])
print("Held-out comparison (8 of 32 clients), same split, same metric, this run:")
print(comparison.round(3).to_string())
print()

# --- 1d. the playbook tables ---
print("Ranked action counts across the whole queue:")
print(queue["action_label"].value_counts().to_string())
print()
print("Reason code counts:")
print(queue["reason_code"].value_counts().to_string())
print()
print("Archetype -> action map (n, declining rate, action):")
arch_order = ["zero_click_visible", "ctr_lag_visible", "stale_tail_visible",
              "visible_opportunity", "buried_visible", "quiet_low_volume"]
arch_tab = queue.groupby("archetype", observed=True).agg(
    n=("archetype", "size"), rate=("is_declining_label", "mean"),
    action=("action_label", "first")).reindex(arch_order).dropna(subset=["n"]).round(3)
print(arch_tab.to_string())
print()

full = {"P@10": prec_at_k(queue["is_declining_label"], 10), "P@50": prec_at_k(queue["is_declining_label"], 50)}
print(f"Full-slice queue precision (all 30k pages): P@10 {full['P@10']:.3f} | P@50 {full['P@50']:.3f}  "
      f"(base {queue['is_declining_label'].mean():.3f})")

Held-out comparison (8 of 32 clients), same split, same metric, this run:
                         P@10   P@50
Base rate               0.634  0.634
Random Forest           0.800  0.880
Logistic Regression     0.800  0.860
Transparent rule (w04)  0.900  0.920

Ranked action counts across the whole queue:
action_label
monitor                   11248
refresh_and_review_ctr    10730
watch_and_monitor          7159
refresh                     863

Reason code counts:
reason_code
low_traffic_monitor    11248
ctr_lag_visible        10730
visible_opportunity     7159
stale_visible            863

Archetype -> action map (n, declining rate, action):
                         n   rate                  action
archetype                                                
zero_click_visible    1826  0.762  refresh_and_review_ctr
ctr_lag_visible       8904  0.609  refresh_and_review_ctr
stale_tail_visible     863  0.477                 refresh
visible_opportunity   1612  0.488       watch_and_monitor
bur

### What the ranked queue actually looks like (top of the list)

The rows below are the top of the queue from **this run** — score, reason code, archetype, and
the label shown for evaluation only. The top of the list is dominated by the CONFIRMED
`ctr_lag_visible` reason, and the advisory quirk flag (`is_quirk`, zero-CTR + stable/up trend)
is set from the very top — exactly why section 3 makes human review mandatory.

In [3]:
print("The queue's top 10 (label is evaluation-only, never an input to the score):")
cols = ["rank", "score", "action_label", "reason_code", "archetype", "is_quirk",
        "impressions_90d", "ctr", "avg_position", "days_since_last_update", "is_declining_label"]
print(queue.head(10)[cols].round(3).to_string(index=False))
print()
print("Top-50 composition by reason code:")
print(queue.head(50)["reason_code"].value_counts().to_string())
print(f"Top-50 declining rate: {prec_at_k(queue['is_declining_label'], 50):.3f} "
      f"(full-slice base {queue['is_declining_label'].mean():.3f})")

The queue's top 10 (label is evaluation-only, never an input to the score):
 rank  score           action_label     reason_code          archetype  is_quirk  impressions_90d  ctr  avg_position  days_since_last_update  is_declining_label
    1  0.913 refresh_and_review_ctr ctr_lag_visible zero_click_visible         0           208678  0.0           9.7                     104                   1
    2  0.902 refresh_and_review_ctr ctr_lag_visible zero_click_visible         0            16786  0.0           5.6                     104                   1
    3  0.901 refresh_and_review_ctr ctr_lag_visible zero_click_visible         0            16156  0.0           9.0                     104                   1
    4  0.891 refresh_and_review_ctr ctr_lag_visible zero_click_visible         0             7732  0.0           8.3                     104                   1
    5  0.889 refresh_and_review_ctr ctr_lag_visible zero_click_visible         1             7087  0.0          16.0   

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Who.** A content editor or SEO strategist at a site like the 32 pseudonymized clients in this
slice, working with a fixed review budget (≈50 pages per day or week).

**What for.** (1) A daily review queue: open the top-K pages in rank order and decide on an
edit. (2) A triage map: the reason code and archetype say *why* a page is there, so the human
look is faster. (3) The raw material for the paper's recommendations section — the queue,
metrics and figures exported in section 5 are exactly the artifacts that section is built from.

**Cadence.** The queue is a trailing-90-day snapshot; it is valid until the next export. Treat
it as a weekly/monthly report, not a live feed.

**Where it stops being valid (the limits, stated plainly):**

- **Single snapshot, cross-sectional design.** Every association is concurrent: the score and
  the label share the same 90-day window. Nothing predicts a future state, and nothing says an
  edit will *cause* recovery.
- **The label is a proxy.** `is_declining_label = (trend_direction == "down")`, a 30d-vs-prev-30d
  impression change — not an observed business outcome.
- **Quiet pages carry no measurable signal.** Below 300 impressions the queue says "monitor";
  that is an honest "we cannot see it yet", not a clean bill of health.
- **38% of flagged pages are not declining.** The queue is a screen, not a verdict.
- **32 clients only.** The held-out set is 8 clients; treat the numbers as decision-support for
  clients like these, not a general guarantee.
- **No causal or financial claim is licensed.** Refreshes were not run, so "will recover
  traffic" or "saves N hours" would exceed the evidence (w06 rewrite).
- **Non-production.** This is a research artifact for the paper, not a deployed pipeline.

In [4]:
# 2a. the over-flag bound: how many flagged pages are NOT declining?
flagged = queue["action_label"].isin(["refresh_and_review_ctr", "refresh"])
n_flagged = int(flagged.sum())
n_not_dec = int((flagged & (queue["is_declining_label"] == 0)).sum())
print(f"Pages flagged for an edit (refresh_*): {n_flagged:,} | not declining: {n_not_dec:,} "
      f"({n_not_dec/n_flagged*100:.0f}% of the flag population)")
print()

# 2b. coverage: how much of the queue is below the volume where a signal can be measured?
quiet = int((queue["impressions_90d"] < 300).sum())
print(f"Quiet pages (<300 impressions): {quiet:,} ({quiet/len(queue)*100:.0f}% of queue) "
      f"-> 'monitor' means 'no signal yet'")
print()

# 2c. the false-alarm population: zero-CTR at page 1-2 with a stable/up trend (advisory flag)
top50 = queue.head(50)
zc = queue[queue["archetype"] == "zero_click_visible"]
print(f"zero_click_visible (page 1-2, CTR=0): n = {len(zc):,}, declining rate {zc['is_declining_label'].mean():.3f}")
n_quirk = int(queue["is_quirk"].sum())
print(f"zero_ctr_stable_quirk (advisory flag inside it, trend stable/up): n = {n_quirk:,}")
print(f"  in the top-50: {int(top50['is_quirk'].sum())}/50 -> keep the high score but verify before editing")
print()

# 2d. window truth
print("Window: the label is a 30d-vs-prev-30d trend inside the same trailing-90-day snapshot as the features.")
print("=> claims are concurrent associations (observed, directional), NOT forward predictions.")

Pages flagged for an edit (refresh_*): 11,593 | not declining: 4,363 (38% of the flag population)

Quiet pages (<300 impressions): 11,248 (37% of queue) -> 'monitor' means 'no signal yet'

zero_click_visible (page 1-2, CTR=0): n = 1,826, declining rate 0.762
zero_ctr_stable_quirk (advisory flag inside it, trend stable/up): n = 415
  in the top-50: 10/50 -> keep the high score but verify before editing

Window: the label is a 30d-vs-prev-30d trend inside the same trailing-90-day snapshot as the features.
=> claims are concurrent associations (observed, directional), NOT forward predictions.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

**The review contract — what the human checks on every top-K page before editing:**

1. **Is the CTR gap real?** Open the page. If impressions are navigational/branded or from a
   bot, zero clicks may be the *correct* outcome, not a failure (the most common trap in this
   data).
2. **Is the position real?** A page that just climbed may be averaging an older, low-traffic
   period. Check the recent trend, not the 90-day mean.
3. **Is the decline demand-driven?** If the target query's season collapsed, the fix is not a
   refresh — there is simply no search demand this season.
4. **Is it stale for a reason?** A page left untouched may be intentionally evergreen; 104+ days
   is a cue to *check*, not a sentence to rewrite.
5. **Trust the archetype over the score** at the top: a `zero_ctr_stable_quirk` row keeps its
   high score but gets extra skepticism before any edit.

**The no-go list — what should NEVER be automated (stated plainly):**

- ❌ **No auto-refresh / auto-edit.** The score ranks pages; it does not write copy. Refreshing is
  an editorial decision made on an open page.
- ❌ **No auto-prune / no auto-delete.** Deletion is irreversible, and this design measured no
  outcome after an edit.
- ❌ **No auto-decision on a zero-CTR page.** `ctr = 0` is frequently a tracking or query-type
  artifact; a human must verify before anything is touched.
- ❌ **No "top-50 equals act" rule.** The top-K is a *review* batch: the full-slice top-50 is
  ~80% declining, so ~10 of 50 are healthy pages to **skip**, not "fix".
- ❌ **No score-only delegation.** The learned models do not beat the rule on held-out clients,
  so neither the score nor a model may replace the rule's transparent reasons.
- ❌ **No causal wording in any report.** The playbook's sentences stay decision-support
  (observed / measured / directional).

In [5]:
# 3a. top-5 with the human check, produced from this run's rows
caveats = {
    "zero_click_visible": "would be wrong if ctr=0 is a tracking gap, bots, or navigational queries -- verify before editing",
    "zero_ctr_stable_quirk": "would be wrong if the query simply never clicks -- the stable label is the check, not the score",
    "ctr_lag_visible": "would be wrong if the position just climbed and the 90-day CTR averages an older period",
    "stale_tail_visible": "would be wrong if the page is intentionally evergreen -- 104+ days is a cue to check, not rewrite",
    "visible_opportunity": "would be wrong if a demand collapse is coming -- the flag says watch, not fix",
    "quiet_low_volume": "would be wrong if the page is actually broken but too small to show it -- re-measure when volume grows",
}
print("Top-5 of the queue, each with the human check it needs:")
for _, row in queue.head(5).iterrows():
    if row["is_quirk"] == 1:
        caveat = "would be wrong if the query simply never clicks -- the stable/up trend is the check, not the score"
    else:
        caveat = caveats.get(row["archetype"], "")
    print(f"  rank {int(row['rank']):>2} | {row['action_label']:<22} | {row['archetype']:<22} | "
          f"score {row['score']:.3f} | {caveat}")
print()

# 3b. the no-go arithmetic: healthy pages inside the top-50 to SKIP, not fix
top50 = queue.head(50)
n_decl = int(top50["is_declining_label"].sum())
print(f"No-go check (full-slice top-50): {n_decl}/50 declining -> {50 - n_decl} are healthy pages to SKIP, not fix.")
print()

# 3c. concrete no-go rows: the quirk population inside the top-50 (verify, don't edit)
quirk50 = top50[top50["is_quirk"] == 1]
cols = ["rank", "archetype", "action_label", "ctr", "avg_position", "impressions_90d", "is_declining_label"]
print("Example rows where the human rule overrides the score (quirk flag set, in the top-50):")
print(quirk50[cols].round(3).to_string(index=False))

Top-5 of the queue, each with the human check it needs:
  rank  1 | refresh_and_review_ctr | zero_click_visible     | score 0.913 | would be wrong if ctr=0 is a tracking gap, bots, or navigational queries -- verify before editing
  rank  2 | refresh_and_review_ctr | zero_click_visible     | score 0.902 | would be wrong if ctr=0 is a tracking gap, bots, or navigational queries -- verify before editing
  rank  3 | refresh_and_review_ctr | zero_click_visible     | score 0.901 | would be wrong if ctr=0 is a tracking gap, bots, or navigational queries -- verify before editing
  rank  4 | refresh_and_review_ctr | zero_click_visible     | score 0.891 | would be wrong if ctr=0 is a tracking gap, bots, or navigational queries -- verify before editing
  rank  5 | refresh_and_review_ctr | zero_click_visible     | score 0.889 | would be wrong if the query simply never clicks -- the stable/up trend is the check, not the score

No-go check (full-slice top-50): 40/50 declining -> 10 are healthy pages

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

The queue is only as good as the snapshot it was built on. These are cheap, light signals —
research-grade monitoring, not a production MLOps stack.

| Watch | Current baseline (this run) | Trigger that says "the queue is stale" | Action |
|---|---|---|---|
| Queue precision on a fresh export | P@50 = 0.92 held-out / 0.80 full slice | Next validated P@50 < base rate + 0.15 | Re-audit the signals before trusting the queue |
| Model vs rule | rule 0.92 > RF 0.88 (P@50) | A retrained RF beats the rule by ≥ +0.05 P@50 on a fresh grouped holdout | Switch the queue to the model — same split, same leak guard |
| Signal relationships | CTR-at-position monotonic; only 104+ day staleness elevated | CTR-vs-position flattens, or fresh pages decline at the tail rate | Re-run the w04 audits before rebuilding the queue |
| Reason-code mix | ctr_lag 10.7k / stale 0.9k / watch 7.2k / monitor 11.2k | Any code's share shifts >20% between exports | The data-generating process changed (tracking, content mix, season) |
| Data freshness | trailing-90-day snapshot | >90 days since export | Rebuild the queue on a fresh export |
| Label balance | 0.542 full / 0.634 held-out | Base rate drifts >0.05 | The page mix changed — re-frame the target |
| Visible zero-CTR volume | 1,826 on page 1-2 | A spike in zero-CTR flags | Check for a tracking change before "fixing" anything |

**Retrain trigger in one sentence.** Rebuild the queue when (a) a fresh export arrives, or
(b) the reason-code mix or signal relationships shift as above, or (c) the learned models
finally earn their complexity on a fresh held-out grouped split. Until one of those fires, the
current queue — and the rule behind it — is the reference.

In [6]:
# --- current baselines for the monitoring table (computed in this run) ---
print("Monitoring baselines (this run):")
print(f"  Full-slice P@50: {prec_at_k(queue['is_declining_label'], 50):.3f}  "
      f"(base {queue['is_declining_label'].mean():.3f})")
print(f"  Held-out rule P@50: {results['Transparent rule (w04)']['P@50']:.3f} | "
      f"RF P@50: {results['Random Forest']['P@50']:.3f} | "
      f"LR P@50: {results['Logistic Regression']['P@50']:.3f}  (test base {y[test_mask].mean():.3f})")
print(f"  Reason-code mix: {queue['reason_code'].value_counts().to_dict()}")
z = int(((df['impressions_90d'] >= 300) & on12 & (df['ctr'] == 0)).sum())
print(f"  Visible zero-CTR pages (page 1-2): n = {z:,}  (a spike = tracking-change smell)")
vis = df[df["impressions_90d"] >= 300]
fresh_rate = vis.loc[vis["days_since_last_update"] <= 30, "is_declining_label"].mean()
tail_rate = vis.loc[vis["days_since_last_update"] >= 104, "is_declining_label"].mean()
print(f"  Staleness sanity: fresh (<=30d) declining rate {fresh_rate:.3f} vs 104+ day tail {tail_rate:.3f}")
print("    -> if fresh pages start declining at the tail rate, the refresh insight has broken.")
print()
print("RETRAIN TRIGGERS (copy to the paper):")
print("  1. Fresh export: >90 days since export -> rebuild the queue.")
print("  2. Precision drop: next validated P@50 < base rate + 0.15.")
print("  3. Signal flip: staleness becomes monotonic, or CTR-at-position flattens -> re-audit (w04 tests).")
print("  4. Mix drift: any reason code's share shifts >20% between exports.")
print("  5. Model earns it: RF P@50 >= rule P@50 + 0.05 on a fresh grouped holdout -> consider switching.")

Monitoring baselines (this run):
  Full-slice P@50: 0.800  (base 0.542)
  Held-out rule P@50: 0.920 | RF P@50: 0.880 | LR P@50: 0.860  (test base 0.634)
  Reason-code mix: {'low_traffic_monitor': 11248, 'ctr_lag_visible': 10730, 'visible_opportunity': 7159, 'stale_visible': 863}
  Visible zero-CTR pages (page 1-2): n = 1,826  (a spike = tracking-change smell)
  Staleness sanity: fresh (<=30d) declining rate 0.581 vs 104+ day tail 0.620
    -> if fresh pages start declining at the tail rate, the refresh insight has broken.

RETRAIN TRIGGERS (copy to the paper):
  1. Fresh export: >90 days since export -> rebuild the queue.
  2. Precision drop: next validated P@50 < base rate + 0.15.
  3. Signal flip: staleness becomes monotonic, or CTR-at-position flattens -> re-audit (w04 tests).
  4. Mix drift: any reason code's share shifts >20% between exports.
  5. Model earns it: RF P@50 >= rule P@50 + 0.05 on a fresh grouped holdout -> consider switching.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

| File | Contents | In git? |
|---|---|---|
| `work/outputs/action_playbook_queue.csv` | full ranked queue: rank, score, reason_code, action_label, archetype + supporting columns | No — the CI leak-guard blocks data CSVs; the notebook regenerates it |
| `work/outputs/w07_playbook_metrics.json` | the numbers this playbook's claims trace back to | Yes — the receipts |
| `work/figures/w07_precision_at_k.png` | held-out Precision@K: rule vs RF vs base rate | Yes |
| `work/figures/w07_staleness_insight.png` | the decay/refresh insight (staleness buckets vs declining rate) | Yes |
| `work/figures/w07_ctr_insight.png` | the CTR-at-position signal (the CONFIRMED finding) | Yes |
| `work/figures/w07_archetype_actions.png` | the archetype→action map with n and declining rates | Yes |

The CSV stays out of git by design (the notebook regenerates it on every run); the figures and
metrics JSON are committed so the paper's charts and numbers trace back to this notebook.

In [7]:
out_dir = root / "work" / "outputs"
fig_dir = root / "work" / "figures"
out_dir.mkdir(parents=True, exist_ok=True)
fig_dir.mkdir(parents=True, exist_ok=True)

# --- the ranked queue CSV (gitignored by design; regenerated on every run) ---
queue_cols = ["rank", "content_id", "client_id", "score", "reason_code", "action_label", "archetype",
              "is_quirk", "impressions_90d", "clicks_90d", "ctr", "avg_position",
              "days_since_last_update", "content_age_days", "is_declining_label"]
queue[queue_cols].to_csv(out_dir / "action_playbook_queue.csv", index=False)
print("Wrote", out_dir / "action_playbook_queue.csv", f"({len(queue):,} rows)")

Wrote E:\FlyRank Ai\flyrank-assignment1\work\outputs\action_playbook_queue.csv (30,000 rows)


In [8]:
# --- figure 1: held-out Precision@K (why this queue works) ---
ks = [10, 20, 50, 100, 200]
rf_rk = pd.DataFrame({"y": y[test_mask].values, "p": results["Random Forest"]["proba"]}).sort_values("p", ascending=False)
p_rule = [prec_at_k(base_y["y"], k) for k in ks]
p_rf = [prec_at_k(rf_rk["y"], k) for k in ks]
base = y[test_mask].mean()

fig, ax = plt.subplots(figsize=(7, 4.2))
ax.plot(ks, p_rule, "o-", color="#1f77b4", label=f"Transparent rule (P@50 {p_rule[2]:.2f})")
ax.plot(ks, p_rf, "s--", color="#d62728", label=f"Random Forest (P@50 {p_rf[2]:.2f})")
ax.axhline(base, color="gray", ls=":", label=f"Test base rate {base:.3f}")
ax.set_xlabel("K (top-K pages reviewed)")
ax.set_ylabel("Precision@K (share declining)")
ax.set_title("Held-out clients (8 of 32): rule vs learned model, same split, same metric")
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(fig_dir / "w07_precision_at_k.png", dpi=150)
plt.close(fig)
print("Wrote", fig_dir / "w07_precision_at_k.png")

Wrote E:\FlyRank Ai\flyrank-assignment1\work\figures\w07_precision_at_k.png


In [9]:
# --- figure 2: the decay/refresh insight (staleness is NOT a general risk signal) ---
vis2 = df[df["impressions_90d"] >= 300].copy()
bins = pd.cut(vis2["days_since_last_update"], [0, 30, 90, 103, 179, 1e6],
              labels=["0-30", "31-90", "91-103", "104-179", "180+"])
st = vis2.groupby(bins, observed=True)["is_declining_label"].agg(n="size", rate="mean")

fig, ax = plt.subplots(figsize=(7, 4.4))
labels = [str(x) for x in st.index]
cols = ["#2ca02c", "#2ca02c", "#7f7f7f", "#d62728", "#d62728"]
ax.bar(labels, st["rate"].values, color=cols, alpha=0.85)
for i, (n, r) in enumerate(zip(st["n"].values, st["rate"].values)):
    ax.text(i, r + 0.02, f"n={int(n):,}\n{r:.2f}", ha="center", fontsize=9)
ax.axhline(vis2["is_declining_label"].mean(), color="gray", ls=":",
           label=f"visible base {vis2['is_declining_label'].mean():.2f}")
ax.set_ylim(0, 1)
ax.set_xlabel("Days since last update")
ax.set_ylabel("Declining rate")
ax.set_title("Decay/refresh insight: only the 104+ day tail is elevated (n printed)")
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(fig_dir / "w07_staleness_insight.png", dpi=150)
plt.close(fig)
print("Wrote", fig_dir / "w07_staleness_insight.png")

Wrote E:\FlyRank Ai\flyrank-assignment1\work\figures\w07_staleness_insight.png


In [10]:
# --- figure 3: the CONFIRMED signal (CTR at page 1-2) ---
pos12 = df[(df["avg_position"] > 0) & (df["avg_position"] <= 20)].copy()
band = pd.cut(pos12["ctr"], [0, 0.1, 0.5, 1.5, 100], labels=["<0.1", "0.1-0.5", "0.5-1.5", ">1.5"])
ct = pos12.groupby(band, observed=True)["is_declining_label"].agg(n="size", rate="mean")

fig, ax = plt.subplots(figsize=(7, 4.4))
ax.bar([str(x) for x in ct.index], ct["rate"].values, color="#1f77b4", alpha=0.85)
for i, (n, r) in enumerate(zip(ct["n"].values, ct["rate"].values)):
    ax.text(i, r + 0.02, f"n={int(n):,}\n{r:.2f}", ha="center", fontsize=9)
ax.axhline(pos12["is_declining_label"].mean(), color="gray", ls=":",
           label=f"pos 1-20 base {pos12['is_declining_label'].mean():.2f}")
ax.set_ylim(0, 1)
ax.set_xlabel("CTR band (rate columns are x100 percentages)")
ax.set_ylabel("Declining rate")
ax.set_title("The CONFIRMED signal: declining rate rises as CTR falls at page 1-2")
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(fig_dir / "w07_ctr_insight.png", dpi=150)
plt.close(fig)
print("Wrote", fig_dir / "w07_ctr_insight.png")

Wrote E:\FlyRank Ai\flyrank-assignment1\work\figures\w07_ctr_insight.png


In [11]:
# --- figure 4: archetype -> action map (bar = pages, label = declining rate) ---
action_color = {"refresh_and_review_ctr": "#d62728", "refresh": "#ff7f0e",
                "watch_and_monitor": "#1f77b4", "monitor": "#7f7f7f"}
arch_tab4 = queue.groupby("archetype", observed=True).agg(
    n=("archetype", "size"), rate=("is_declining_label", "mean"),
    action=("action_label", "first")).reindex(arch_order).dropna(subset=["n"]).reset_index()

fig, ax = plt.subplots(figsize=(8, 4.6))
ypos = np.arange(len(arch_tab4))
ax.barh(ypos, arch_tab4["n"].values,
        color=[action_color[a] for a in arch_tab4["action"].values], alpha=0.85)
for i, (n, r) in enumerate(zip(arch_tab4["n"].values, arch_tab4["rate"].values)):
    ax.text(n * 1.02, i, f"n={int(n):,} | declining {r:.2f}", va="center", fontsize=9)
ax.set_yticks(ypos, arch_tab4["archetype"].values)
ax.invert_yaxis()
ax.set_xlabel("Pages in this slice")
ax.set_title("Archetype -> action map (color = action, label = declining rate)")
ax.set_xlim(0, arch_tab4["n"].max() * 1.3)
fig.tight_layout()
fig.savefig(fig_dir / "w07_archetype_actions.png", dpi=150)
plt.close(fig)
print("Wrote", fig_dir / "w07_archetype_actions.png")

Wrote E:\FlyRank Ai\flyrank-assignment1\work\figures\w07_archetype_actions.png


In [12]:
# --- the metrics receipt (committed; the paper's numbers trace back to it) ---
metrics = {
    "rows": int(len(queue)),
    "clients": int(df["client_id"].nunique()),
    "held_out_clients": int(len(holdout)),
    "base_declining_rate_full": round(float(queue["is_declining_label"].mean()), 4),
    "base_declining_rate_test": round(float(y[test_mask].mean()), 4),
    "full_slice_precision": {"P@10": round(float(full["P@10"]), 4), "P@50": round(float(full["P@50"]), 4)},
    "held_out_precision": {k: {"P@10": round(float(v["P@10"]), 4), "P@50": round(float(v["P@50"]), 4)}
                           for k, v in results.items()},
    "reason_code_counts": {k: int(v) for k, v in queue["reason_code"].value_counts().items()},
    "action_label_counts": {k: int(v) for k, v in queue["action_label"].value_counts().items()},
    "archetype_counts": {k: int(v) for k, v in queue["archetype"].value_counts().items()},
    "zero_ctr_stable_quirk_n": int(queue["is_quirk"].sum()),
    "decay_refresh_insight": {
        "staleness_verdict": "MIXED - only the 104+ day tail is elevated",
        "fresh_30d_rate": round(float(fresh_rate), 4),
        "tail_104d_rate": round(float(tail_rate), 4),
        "zero_click_visible_rate": round(float(queue.loc[queue["archetype"] == "zero_click_visible", "is_declining_label"].mean()), 4),
    },
    "no_go": ["no auto-refresh/auto-edit", "no auto-prune/delete",
              "no auto-decision on zero-CTR", "no score-only delegation", "no causal wording"],
    "queue_csv": "work/outputs/action_playbook_queue.csv",
    "metrics_json": "work/outputs/w07_playbook_metrics.json",
    "figures": ["work/figures/w07_precision_at_k.png", "work/figures/w07_staleness_insight.png",
                "work/figures/w07_ctr_insight.png", "work/figures/w07_archetype_actions.png"],
}
with open(out_dir / "w07_playbook_metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2, sort_keys=True)
print("Wrote", out_dir / "w07_playbook_metrics.json")
print()
print("Exports complete. Queue rows:", len(queue), "| figures:", len(metrics["figures"]))
for fp in [out_dir / "action_playbook_queue.csv"] + [fig_dir / Path(f).name for f in metrics["figures"]] + [out_dir / "w07_playbook_metrics.json"]:
    print("  exists:", fp.exists(), "|", fp)

Wrote E:\FlyRank Ai\flyrank-assignment1\work\outputs\w07_playbook_metrics.json

Exports complete. Queue rows: 30000 | figures: 4
  exists: True | E:\FlyRank Ai\flyrank-assignment1\work\outputs\action_playbook_queue.csv
  exists: True | E:\FlyRank Ai\flyrank-assignment1\work\figures\w07_precision_at_k.png
  exists: True | E:\FlyRank Ai\flyrank-assignment1\work\figures\w07_staleness_insight.png
  exists: True | E:\FlyRank Ai\flyrank-assignment1\work\figures\w07_ctr_insight.png
  exists: True | E:\FlyRank Ai\flyrank-assignment1\work\figures\w07_archetype_actions.png
  exists: True | E:\FlyRank Ai\flyrank-assignment1\work\outputs\w07_playbook_metrics.json


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Ranked actions with reason codes (priority-ordered, each backed by the w04 verdict)
- [x] Archetype → action mapping with counts and declining rates computed in this run
- [x] Decay/refresh insight stated honestly (staleness MIXED; 104+ day tail only)
- [x] Intended use and limits explicit (who / what / cadence / where it stops being valid)
- [x] Human review contract + the no-go list (what should NEVER be automated)
- [x] Cost/value thinking with honest numbers (P@50 0.92 held-out; ~1 in 3 flagged not declining)
- [x] Monitoring / retrain triggers with current baselines computed in this run
- [x] Queue CSV exported to `work/outputs/` (gitignored by design; regenerated); figures committed to `work/figures/`; metrics JSON committed
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [13]:
# validation marker — this cell runs last and stays empty by design